In [1]:
!pip3 install -q -U docling docling-core
!pip3 install -q -U langchain langchain-classic langchain-milvus langchain-huggingface langchain-openai langchain-community
!pip3 install -q -U pymilvus sentence-transformers rank_bm25
!pip3 install -q -U pillow


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pickle
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.base_models import InputFormat

SOURCE_PATH = "HR Policy Manual 2023.pdf"  # swap for any PDF/DOCX/PPTX path
CACHE_PATH = Path("document.cache.pkl")

if CACHE_PATH.exists():
    doc = pickle.loads(CACHE_PATH.read_bytes())
    print(f"Loaded cached parse. Pages: {len(doc.pages)}")
else:
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = False  # set True only if PDF is scanned/image-based — OCR is the slowest step
    pipeline_options.do_table_structure = True
    pipeline_options.table_structure_options.mode = TableFormerMode.ACCURATE
    pipeline_options.generate_picture_images = False  # not used downstream (chunker drops pure-image blocks anyway)

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    result = converter.convert(SOURCE_PATH)
    doc = result.document
    CACHE_PATH.write_bytes(pickle.dumps(doc))
    print(f"Parsed and cached. Pages: {len(doc.pages)}")

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded cached parse. Pages: 208


In [3]:
data = doc.export_to_dict()
n_tables = len(data.get("tables", []))
n_pictures = len(data.get("pictures", []))
n_texts = len(data.get("texts", []))
print(f"Tables: {n_tables} | Pictures/figures: {n_pictures} | Text blocks: {n_texts}")

Tables: 116 | Pictures/figures: 16 | Text blocks: 3283


In [4]:
import json
from docling_core.transforms.chunker import HybridChunker
from docling_core.types.doc import TableItem, PictureItem

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"  # base: ~3x faster than bge-large, still top-tier MTEB retrieval
CHUNKS_CACHE = Path("chunks.cache.json")

if CHUNKS_CACHE.exists():
    cached = json.loads(CHUNKS_CACHE.read_text())
    chunk_texts, chunk_metas = cached["texts"], cached["metas"]
else:
    chunker = HybridChunker(
        tokenizer=EMBED_MODEL_NAME,
        max_tokens=512,
    )

    raw_chunks = list(chunker.chunk(doc))

    chunk_texts = []
    chunk_metas = []
    for c in raw_chunks:
        text = chunker.contextualize(c)  # prepends heading context, replaces deprecated .serialize()
        doc_items = c.meta.doc_items or []
        has_table = any(isinstance(item, TableItem) for item in doc_items)
        has_picture = any(isinstance(item, PictureItem) for item in doc_items)
        page = doc_items[0].prov[0].page_no if doc_items and doc_items[0].prov else None

        chunk_texts.append(text)
        chunk_metas.append({
            "page": page,
            "headings": " > ".join(c.meta.headings) if c.meta.headings else "",
            "has_table": has_table,
            "has_picture": has_picture,
        })

    CHUNKS_CACHE.write_text(json.dumps({"texts": chunk_texts, "metas": chunk_metas}))

print(f"Total chunks: {len(chunk_texts)}")
print(f"Chunks with tables: {sum(m['has_table'] for m in chunk_metas)}")
print(f"Chunks with figures: {sum(m['has_picture'] for m in chunk_metas)}")

Total chunks: 457
Chunks with tables: 102
Chunks with figures: 0


In [5]:
import torch
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Embedding device: {device}")

embeddings = HuggingFaceBgeEmbeddings(
    model_name=EMBED_MODEL_NAME,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 64},
    query_instruction="Represent this sentence for searching relevant passages: ",
)

sample_vec = embeddings.embed_query("test")
print(f"Embedding dim: {len(sample_vec)}")

Embedding device: mps


/var/folders/vc/47nq4gp93_b0p5h81kh5xmtm0000gp/T/ipykernel_29874/1818080843.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceBgeEmbeddings
/var/folders/vc/47nq4gp93_b0p5h81kh5xmtm0000gp/T/ipykernel_29874/1818080843.py:12: LangChainDeprecationWarning: The class `HuggingFaceBgeEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceBgeEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11333.80it/s]

Embedding dim: 768


In [6]:
from pymilvus import MilvusClient

MILVUS_URI = "http://localhost:19530"
milvus_client = MilvusClient(uri=MILVUS_URI)
print("Connected. Existing collections:", milvus_client.list_collections())

Connected. Existing collections: ['hr_policy_manual', 'hr_policy_manual_modern_rag']


In [7]:
from langchain_core.documents import Document

lc_documents = [
    Document(page_content=chunk_texts[i], metadata=chunk_metas[i])
    for i in range(len(chunk_texts))
]
print(f"Total LangChain documents: {len(lc_documents)}")

Total LangChain documents: 457


In [8]:
from langchain_milvus import Milvus

COLLECTION_NAME = "hr_policy_manual_modern_rag"

vectorstore = Milvus.from_documents(
    documents=lc_documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    connection_args={"uri": MILVUS_URI},
    index_params={"index_type": "HNSW", "metric_type": "COSINE", "params": {"M": 16, "efConstruction": 200}},
    search_params={"metric_type": "COSINE", "params": {"ef": 64}},
    drop_old=True,
)

print("Vectorstore ready:", COLLECTION_NAME)

Vectorstore ready: hr_policy_manual_modern_rag


In [9]:
from langchain_community.retrievers import BM25Retriever
try:
    from langchain_classic.retrievers import EnsembleRetriever  # LangChain >= 1.0
except ModuleNotFoundError:
    from langchain.retrievers import EnsembleRetriever  # LangChain < 1.0

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

bm25_retriever = BM25Retriever.from_documents(lc_documents)
bm25_retriever.k = 10

hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.6, 0.4],
)

test_hits = hybrid_retriever.invoke("casual leave entitlement")
print(f"Hybrid retriever returned {len(test_hits)} docs")

Hybrid retriever returned 17 docs


In [10]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
try:
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
    from langchain_classic.retrievers import ContextualCompressionRetriever
except ModuleNotFoundError:
    from langchain.retrievers.document_compressors import CrossEncoderReranker
    from langchain.retrievers import ContextualCompressionRetriever

cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-base", model_kwargs={"device": device})
reranker = CrossEncoderReranker(model=cross_encoder, top_n=5)

retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=hybrid_retriever,
)

reranked_hits = retriever.invoke("casual leave entitlement")
for d in reranked_hits:
    print(f"[page {d.metadata.get('page')}] {d.metadata.get('headings')}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8980.42it/s]

[page 79] 5.1 LEAVE TYPE 1: CASUAL LEAVE
[page 131] LEAVE TRAVEL CONCESSION
[page 7] CONTENTS
[page 83] 5.7 LEAVE TYPE 7: PATERNITY LEAVE
[page 81] 5.4 LEAVE TYPE 4: COMMUTED LEAVE


In [11]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# export OPENROUTER_API_KEY="your-key" in your shell before starting Jupyter
llm = ChatOpenAI(
    model="anthropic/claude-sonnet-4.5",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_tokens=500,
)

prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below. Cite the page number(s) you used.
If a fact comes from a table, say so explicitly. If the answer isn't in the context, say so.

Context:
{context}

Question: {question}
"""
)

def format_docs(docs):
    parts = []
    for d in docs:
        tag = "TABLE" if d.metadata.get("has_table") else ("FIGURE" if d.metadata.get("has_picture") else "TEXT")
        parts.append(f"[{tag} | page {d.metadata.get('page')} | {d.metadata.get('headings')}]\n{d.page_content}")
    return "\n\n---\n\n".join(parts)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 11. Test queries

In [12]:
print(rag_chain.invoke("How many days of casual leave are employees entitled to?"))

According to page 79, section 5.1.1, employees are entitled to **eight days** of casual leave for a calendar year, subject to the condition that not more than five days' casual leave may be allowed at a time.


In [13]:
print(rag_chain.invoke("What is the maximum age for the Chief Financial Officer position?"))

Based on the context provided, the maximum age for the Chief Financial Officer position is **55 years**.

**Source:** Page 22 (table showing recruitment criteria for various posts at IIMA)


In [14]:
print(rag_chain.invoke("What are the different leave categories and their respective day allocations?"))

The context does not provide information about the day allocations for the different leave categories. 

While the table of contents on page 7 lists the different leave types (Casual Leave, Earned Leave, Half Pay Leave, Commuted Leave, Extraordinary Leave, Maternity Leave, and Paternity Leave), and page 82 provides details about Extraordinary Leave policies, none of the provided text includes specific information about the number of days allocated for each leave type.
